# Credit Risk Portfolio Analysis

This notebook analyzes the cleaned credit-risk dataset from a portfolio perspective.

The goal is to understand how default risk varies across borrower and loan characteristics,
and to distinguish between **default frequency** and **monetary exposure**.

The analysis uses the cleaned dataset created in `02_data_cleaning.ipynb`.


## 1. Imports


In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2. Load the Cleaned Dataset

The analysis uses `data/processed/credit_risk_clean.csv`, which is generated by the
data-cleaning notebook.


In [18]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_risk_clean.csv"
)

print(DATA_PATH)
print("File exists:", DATA_PATH.exists())

/Users/marijacolic/PycharmProjects/credit-risk-portfolio-analysis/data/processed/credit_risk_clean.csv
File exists: True


In [19]:
df = pd.read_csv(DATA_PATH)

In [20]:
clean_df = df.copy()

## 3. Portfolio Overview

The following summary provides the size of the portfolio, total and average loan amounts,
average interest rate, and overall default rate.


In [21]:
portfolio_summary = {
    "number_of_loans": len(clean_df),
    "total_loan_amount": clean_df["loan_amnt"].sum(),
    "average_loan_amount": clean_df["loan_amnt"].mean(),
    "average_interest_rate": clean_df["loan_int_rate"].mean(),
    "default_rate": clean_df["loan_status"].mean(),
}

pd.Series(portfolio_summary)

number_of_loans          3.258100e+04
total_loan_amount        3.124313e+08
average_loan_amount      9.589371e+03
average_interest_rate    1.101390e+01
default_rate             2.181640e-01
dtype: float64

## 4. Reusable Risk Summary

The helper function below summarizes applicant count, number of defaults, default rate,
total loan amount, and average loan amount for a selected categorical variable.


In [22]:
def create_risk_summary(
    data: pd.DataFrame,
    group_column: str
) -> pd.DataFrame:
    """Create a default summary for a categorical variable."""

    summary = (
        data.groupby(group_column, dropna=False)
        .agg(
            applicant_count=("loan_status", "size"),
            default_count=("loan_status", "sum"),
            default_rate=("loan_status", "mean"),
            total_loan_amount=("loan_amnt", "sum"),
            average_loan_amount=("loan_amnt", "mean")
        )
        .reset_index()
    )

    summary["default_rate"] *= 100

    return summary.sort_values(
        "default_rate",
        ascending=False
    )

## 5. Default Risk by Loan Grade

This section compares observed default rates across loan grades. Because some higher-risk
grades contain relatively few observations, both the default rate and the number of
applicants should be considered when interpreting the results.


In [23]:
create_risk_summary(clean_df, "loan_grade")

,loan_grade,applicant_count,default_count,default_rate,total_loan_amount,average_loan_amount
6,G,64,63,98.437500,1100525,17195.703125
5,F,241,170,70.539419,3546875,14717.323651
4,E,964,621,64.419087,12450875,12915.845436
3,D,3626,2141,59.045780,39339350,10849.241589
2,C,6458,1339,20.733973,59503125,9213.862651
1,B,10451,1701,16.275954,104462800,9995.483686
0,A,10777,1073,9.956389,92027750,8539.273453


## 6. Default Risk by Loan Purpose

This section compares default rates across different loan purposes.


In [24]:
create_risk_summary(clean_df, "loan_intent")

,loan_intent,applicant_count,default_count,default_rate,total_loan_amount,average_loan_amount
0,DEBTCONSOLIDATION,5212,1490,28.587874,50008550,9594.886800
3,MEDICAL,6071,1621,26.700708,56214925,9259.582441
2,HOMEIMPROVEMENT,3605,941,26.102635,37349675,10360.520111
4,PERSONAL,5521,1098,19.887702,52856800,9573.772867
1,EDUCATION,6453,1111,17.216798,61191725,9482.678599
5,VENTURE,5719,847,14.810282,54809625,9583.777758


## 7. Default Risk by Home Ownership

This section examines whether observed default rates differ across home-ownership groups.


In [25]:
create_risk_summary(clean_df, "person_home_ownership")

,person_home_ownership,applicant_count,default_count,default_rate,total_loan_amount,average_loan_amount
3,RENT,16446,5192,31.569987,145749900,8862.331266
1,OTHER,107,33,30.841121,1184975,11074.532710
0,MORTGAGE,13444,1690,12.570663,142163050,10574.460726
2,OWN,2584,193,7.469040,23333375,9029.943885


## 8. Previous Default History

This comparison shows the observed default rate for borrowers with and without a previous
default recorded in the dataset.


In [26]:
create_risk_summary(
    clean_df,
    "cb_person_default_on_file"
)

,cb_person_default_on_file,applicant_count,default_count,default_rate,total_loan_amount,average_loan_amount
1,Y,5745,2172,37.806789,58158700,10123.359443
0,N,26836,4936,18.393203,254272600,9475.055895


## 9. Derived Portfolio Segments

To support additional segment analysis, borrowers are grouped into age, income, and
loan-to-income bands.


In [27]:
clean_df["age_band"] = pd.cut(
    clean_df["person_age"],
    bins=[0, 24, 34, 44, 54, 64, np.inf],
    labels=[
        "18–24",
        "25–34",
        "35–44",
        "45–54",
        "55–64",
        "65+"
    ]
)

In [ ]:
create_risk_summary(clean_df, "age_band")


In [28]:
clean_df["income_band"] = pd.qcut(
    clean_df["person_income"],
    q=5,
    duplicates="drop"
)

In [ ]:
create_risk_summary(clean_df, "income_band")


In [29]:
clean_df["loan_income_band"] = pd.cut(
    clean_df["loan_percent_income"],
    bins=[0, 0.10, 0.20, 0.30, 0.40, np.inf],
    labels=[
        "0–10%",
        "10–20%",
        "20–30%",
        "30–40%",
        "Above 40%"
    ],
    include_lowest=True
)

In [ ]:
create_risk_summary(clean_df, "loan_income_band")


## 10. Exposure Analysis by Loan Grade

Default rate measures how frequently loans default, but it does not show how much loan
amount is associated with those defaults. The exposure analysis therefore compares total
loan amount with the loan amount linked to defaulted observations.

`defaulted_amount_pct` represents the share of the total loan amount in each grade that is
associated with defaulted loans.


In [30]:
clean_df["defaulted_loan_amount"] = (
    clean_df["loan_amnt"] *
    clean_df["loan_status"]
)

In [ ]:
exposure_summary = (
    clean_df.groupby("loan_grade")
    .agg(
        applicant_count=("loan_status", "size"),
        total_loan_amount=("loan_amnt", "sum"),
        defaulted_loan_amount=("defaulted_loan_amount", "sum"),
        default_rate=("loan_status", "mean")
    )
    .reset_index()
)

exposure_summary["default_rate"] *= 100

exposure_summary["defaulted_amount_pct"] = (
    exposure_summary["defaulted_loan_amount"]
    / exposure_summary["total_loan_amount"]
    * 100
)

exposure_summary.sort_values(
    "defaulted_loan_amount",
    ascending=False
)


## 11. Key Findings

- The portfolio contains **32,581 loan records** with an overall observed default rate of
  approximately **21.82%**.
- Default rates increase substantially across loan grades. Grade A has an observed default
  rate of about **9.96%**, while Grades D, E, F, and G show much higher rates. The very high
  rate for Grade G should be interpreted cautiously because the group contains only
  **64 applicants**.
- **Debt consolidation** has the highest observed default rate among loan purposes at about
  **28.59%**, followed by medical and home-improvement loans.
- Borrowers who **rent** have an observed default rate of about **31.57%**, compared with
  approximately **12.57%** for mortgage holders and **7.47%** for homeowners.
- Borrowers with a **previous default on file** have an observed default rate of about
  **37.81%**, compared with **18.39%** for borrowers without one.
- Portfolio risk should be assessed using both **default rate** and **monetary exposure**.
  A segment can have a lower default rate but still represent a large amount of defaulted
  loan exposure because of its portfolio size.

These findings are descriptive associations in this dataset and should not be interpreted
as causal relationships.
